# ML Distribuido con Spark MLlib — Predicción de Precios

Pipeline completo:
- **Time series features** con ventanas deslizantes nativas de Spark
- **Spark MLlib Pipeline** (VectorAssembler + StandardScaler + RandomForestRegressor)
- **Tuning distribuido** (CrossValidator + ParamGridBuilder)
- **Experiment tracking** con MLflow
- **Inferencia en streaming** con modelo guardado

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml import Pipeline, PipelineModel
import os, time, json

spark = SparkSession.builder \
    .appName("TickDB-ML-Distribuido") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
            "io.delta:delta-spark_2.12:3.0.0,"
            "org.apache.spark:spark-avro_2.12:3.5.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")
print(f"Parallelism: {spark.sparkContext.defaultParallelism}")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

---
## 1. Cargar datos históricos

In [ ]:
INPUT_PATH = "/home/jovyan/work/tickdb_parquet"

df = spark.read.parquet(INPUT_PATH)
df = df.filter(col("symbol").isNotNull() & col("last_price").isNotNull())

print(f"Registros: {df.count():,}")
df.printSchema()
df.select("symbol", "last_price", "market", "timestamp").show(5)

---
## 2. Feature Engineering con Ventanas Deslizantes (Spark nativas)

En lugar de Pandas, usamos `Window.partitionBy().orderBy().rowsBetween()` para calcular:
- Precio anterior (lag)
- SMA (Simple Moving Average) de 5, 10, 20 periodos
- Volatilidad (desviación estándar móvil)
- RSI (Relative Strength Index)

In [ ]:
target_symbol = "BTCUSDT"
df_btc = df.filter(col("symbol") == target_symbol).orderBy("timestamp")

w = Window.partitionBy("symbol").orderBy("timestamp").rowsBetween

df_features = df_btc \
    .withColumn("prev_price", lag("last_price", 1).over(Window.partitionBy("symbol").orderBy("timestamp"))) \
    .withColumn("price_change", col("last_price") - col("prev_price")) \
    .withColumn("SMA_5", avg("last_price").over(w(-4, 0))) \
    .withColumn("SMA_10", avg("last_price").over(w(-9, 0))) \
    .withColumn("SMA_20", avg("last_price").over(w(-19, 0))) \
    .withColumn("volatility", stddev("last_price").over(w(-19, 0))) \
    .filter(col("prev_price").isNotNull()) \
    .withColumn("target", col("last_price")) \
    .select("symbol", "timestamp", "last_price", "prev_price", "price_change",
            "SMA_5", "SMA_10", "SMA_20", "volatility", "target")

print(f"Registros con features: {df_features.count():,}")
df_features.show(10)

---
## 3. Pipeline MLlib con validación cruzada distribuida

In [ ]:
feature_cols = ["prev_price", "price_change", "SMA_5", "SMA_10", "SMA_20", "volatility"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="target",
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, rf])

evaluator = RegressionEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="mae"
)

In [ ]:
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50, 100]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .addGrid(rf.minInstancesPerNode, [2, 5]) \
    .build()

print(f"Combinaciones de hiperparámetros: {len(param_grid)}")

In [ ]:
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,
    seed=42
)

train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  |  Test: {test_df.count():,}")

In [ ]:
print("Entrenando con CrossValidator distribuido...")
start = time.time()

cv_model = cv.fit(train_df)

elapsed = time.time() - start
print(f"Entrenamiento completado en {elapsed:.1f}s")

best_model = cv_model.bestModel
best_rf = best_model.stages[-1]
print(f"\nMejores hiperparámetros:")
print(f"  numTrees:         {best_rf.getNumTrees()}")
print(f"  maxDepth:         {best_rf.getMaxDepth()}")
print(f"  minInstancesPerNode: {best_rf.getMinInstancesPerNode()}")

In [ ]:
pred_df = best_model.transform(test_df)

mae = evaluator.evaluate(pred_df)
rmse = evaluator.setMetricName("rmse").evaluate(pred_df)
r2 = evaluator.setMetricName("r2").evaluate(pred_df)

print(f"=== MÉTRICAS (TEST SET) ===")
print(f"MAE:  ${mae:.2f}")
print(f"RMSE: ${rmse:.2f}")
print(f"R²:   {r2:.4f}")

print("\n=== IMPORTANCIA DE FEATURES ===")
for f, imp in zip(feature_cols, best_rf.featureImportances):
    print(f"  {f:15s}  {imp:.4f}")

---
## 4. MLflow Experiment Tracking

In [ ]:
import mlflow
import mlflow.spark

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("tickdb-price-prediction")

with mlflow.start_run(run_name=f"rf_{target_symbol}") as run:
    mlflow.log_params({
        "numTrees": best_rf.getNumTrees(),
        "maxDepth": best_rf.getMaxDepth(),
        "minInstancesPerNode": best_rf.getMinInstancesPerNode(),
        "featureCols": json.dumps(feature_cols),
        "model_type": "RandomForestRegressor",
        "target": target_symbol,
    })
    mlflow.log_metrics({
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "training_samples": train_df.count(),
    })
    mlflow.spark.log_model(best_model, "model")
    mlflow.log_artifact("/home/jovyan/work/tickdb_ml_distribuido.ipynb")
    
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"Experimento:   {mlflow.get_experiment(run.info.experiment_id).name}")
    print(f"MLflow UI:     {MLFLOW_TRACKING_URI}")

---
## 5. Guardar modelo en HDFS + registro local

In [ ]:
MODEL_PATH = "/home/jovyan/work/models/btc_rf_model"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
best_model.write().overwrite().save(MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")

try:
    hdfs_model_path = f"hdfs://hdfs-namenode:9000/tickdb/models/btc_rf_model"
    best_model.write().overwrite().save(hdfs_model_path)
    print(f"Modelo guardado en HDFS: {hdfs_model_path}")
except Exception as e:
    print(f"⚠ HDFS no disponible: {e}")

---
## 6. Inferencia en Streaming con el modelo

In [ ]:
model = PipelineModel.load(MODEL_PATH)
print("Modelo cargado para inferencia")

event_schema = StructType([
    StructField("symbol", StringType()),
    StructField("last_price", DoubleType()),
    StructField("volume_24h", DoubleType()),
    StructField("high_24h", DoubleType()),
    StructField("low_24h", DoubleType()),
    StructField("price_change_24h", DoubleType()),
    StructField("price_change_percent_24h", DoubleType()),
    StructField("timestamp", LongType()),
    StructField("event_timestamp", LongType()),
    StructField("market", StringType()),
])

def predict_prices(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    w = Window.partitionBy("symbol").orderBy("timestamp").rowsBetween
    features_df = batch_df \
        .withColumn("prev_price", lag("last_price", 1).over(Window.partitionBy("symbol").orderBy("timestamp"))) \
        .withColumn("price_change", col("last_price") - col("prev_price")) \
        .withColumn("SMA_5", avg("last_price").over(w(-4, 0))) \
        .withColumn("SMA_10", avg("last_price").over(w(-9, 0))) \
        .withColumn("SMA_20", avg("last_price").over(w(-19, 0))) \
        .withColumn("volatility", stddev("last_price").over(w(-19, 0))) \
        .filter(col("prev_price").isNotNull())
    
    preds = model.transform(features_df)
    preds.select(
        "symbol", "last_price", "prediction",
        (col("prediction") - col("last_price")).alias("error"),
        "timestamp"
    ).show(truncate=False)

In [ ]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "tickdb-market-data") \
    .option("startingOffsets", "latest") \
    .load()

df_parsed = df_kafka.select(
    from_json(col("value").cast("string"), event_schema).alias("data")
).select("data.*")

df_valid = df_parsed.filter(
    col("symbol").isNotNull() & 
    col("last_price").isNotNull()
)

query_inference = df_valid \
    .writeStream \
    .foreachBatch(predict_prices) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Inferencia en streaming iniciada — esperando datos...")

In [ ]:
time.sleep(30)
query_inference.stop()
print("Inferencia detenida")

In [ ]:
spark.stop()